### RAG(Retrieval-Augmented Generation), 검색 증강 생성
- 생성형 AI가 학습하지 않은 최신 정보나 특정 조직의 내부 데이터를 바탕으로 답변할 수 있도록 돕는 기술이다.
- R (Retrieval): 질문과 관련된 문서를 방대한 데이터베이스에서 검색한다.
- A (Augmentation): 검색된 정보를 질문과 결합하여 프롬프트를 보강한다.
- G (Generation): 보강된 맥락을 바탕으로 최종 답변을 생성한다.

### RAG 개발 순서
1. 문서 불러오기
2. 문서 분할하기
3. 임베딩
4. Vector DB 생성 및 저장
5. 검색기 생성
6. 프롬프트 생성
7. LLM 생성
8. 체인 실행

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

C:\ProgramData\anaconda3\envs\llm\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


In [ ]:
from langchain_community.document_loaders import TextLoader

# 텍스트 로더 생성
loader = TextLoader("경로")

# 문서 로드
docs = loader.load()

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader

# CSV 로더 생성
loader = CSVLoader(file_path="경로")

# 데이터 로드
docs = loader.load()

In [ ]:
from langchain_community.document_loaders import UnstructuredExcelLoader

# Excel 로더 생성
loader = UnstructuredExcelLoader("경로", mode="elements")

# 문서 로드
docs = loader.load()

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader

# 1단계, 문서 로드
FILE_PATH = "./documents/ai커리큘럼.pdf"
loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()

print(f"문서의 페이지 수: {len(docs)}")

문서의 페이지 수: 3


In [9]:
print(docs[0].page_content)

1. 과정 개요 
 
대상: 초보자, 비전공자, 고졸자 
 
목표: 
o PyTorch 기반 머신러닝/딥러닝 실무 능력 배양 
o LangChain + RAG 를 활용한 LLM 문서검색 챗봇 구축 
o SAM2 등 멀티모달 AI 모델 활용 실습 
o MCP(Model Context Protocol) 기반 AI 모델 통합 및 API 설계 
o FastAPI 를 이용한 AI 서비스 상용화 프로젝트 완성 
o 포트폴리오 및 취업 역량 강화 
 
2. 커리큘럼 요약 
월 
주차 
주요 주제 
실습 키워드 
1 월 1~4 주 
Python & PyTorch 기초 
Python, Numpy, Pandas, 선형회귀 
2 월 5~8 주 
PyTorch 딥러닝 기본 
MLP, CNN, 분류 모델 구현 
3 월 9~12 주 LangChain + RAG 활용 
GPT API, LangChain, FAISS, 문서 
검색 
4 월 13~16 주 멀티모달 AI 실습 (SAM2 
포함) 
CLIP, DINOv2, SAM2, Hugging Face 
5 월 17~20 주 MCP + FastAPI 상용화 
MCP 설계, FastAPI API 개발, Docker 
배포 
 
3. 주차별 상세 커리큘럼 
1~2 개월차: 기초 파운데이션 
 
1 주차: Python 기본 문법, Git, VSCode 활용법 
 
2 주차: 함수, 클래스, 자료구조, 알고리즘 기본 
 
3 주차: Numpy, Pandas, 데이터 시각화 기초 
 
4 주차: PyTorch 텐서 기초, 선형 회귀 모델 구현 
 
5 주차: 로지스틱 회귀, 분류 문제 해결 
 
6 주차: MLP 신경망 구성, 손실 함수 이해 
 
7 주차: CNN 이해 및 이미지 분류 실습 
 
8 주차: 중간 프로젝트 - 이미지 또는 텍스트 분류기 완성 
3 개월차: LLM 과 LangChain + RAG


In [11]:
docs[0].__dict__

{'id': None,
 'metadata': {'producer': 'Microsoft® Word 2016',
  'creator': 'Microsoft® Word 2016',
  'creationdate': '2025-08-07T08:56:01+09:00',
  'source': './documents/ai커리큘럼.pdf',
  'file_path': './documents/ai커리큘럼.pdf',
  'total_pages': 3,
  'format': 'PDF 1.5',
  'title': '',
  'author': 'Admin',
  'subject': '',
  'keywords': '',
  'moddate': '2025-08-07T08:56:01+09:00',
  'trapped': '',
  'modDate': "D:20250807085601+09'00'",
  'creationDate': "D:20250807085601+09'00'",
  'page': 0},
 'page_content': '1. 과정 개요 \n\uf0b7 \n대상: 초보자, 비전공자, 고졸자 \n\uf0b7 \n목표: \no PyTorch 기반 머신러닝/딥러닝 실무 능력 배양 \no LangChain + RAG 를 활용한 LLM 문서검색 챗봇 구축 \no SAM2 등 멀티모달 AI 모델 활용 실습 \no MCP(Model Context Protocol) 기반 AI 모델 통합 및 API 설계 \no FastAPI 를 이용한 AI 서비스 상용화 프로젝트 완성 \no 포트폴리오 및 취업 역량 강화 \n \n2. 커리큘럼 요약 \n월 \n주차 \n주요 주제 \n실습 키워드 \n1 월 1~4 주 \nPython & PyTorch 기초 \nPython, Numpy, Pandas, 선형회귀 \n2 월 5~8 주 \nPyTorch 딥러닝 기본 \nMLP, CNN, 분류 모델 구현 \n3 월 9~12 주 LangChain + RAG 활용 \nGPT API, LangChain, FAISS, 

In [13]:
# 2단계, 문서 분할

# chunk_size
# LLM의 기억력과 검색의 정확도를 결정짓는다.
# 너무 작으면 맥락이 잘려서 LLM의 이해도가 떨어지고
# 너무 크면 불필요한 정보가 많이 섞여서 답변의 초점이 흐려진다.
# 500~800자: 3~4문단

# chunk_overlap
# 단락을 분할하면서 생기는 정보의 손실을 최소화하기 위해 연결고리를 만들어준다.
# 피자는 느끼해 난 그래서 싫어
# chunk1: 피자는 느끼해 난 그래서
# chunk2: 느끼해 난 그래서 싫어
# overlap: 느끼해 난 그래서
# 보통 chunk_size의 10~20%정도가 적당하다고 판단한다.

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)
print(f"분할된 청크(조각)의 수: {len(split_documents)}")

분할된 청크(조각)의 수: 5


In [14]:
# 3단계, 임베딩
embeddings = OpenAIEmbeddings()

In [15]:
# 4단계, 벡터 DB
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

In [22]:
vectorstore.similarity_search("PyTorch")[0].page_content

'1. 과정 개요 \n\uf0b7 \n대상: 초보자, 비전공자, 고졸자 \n\uf0b7 \n목표: \no PyTorch 기반 머신러닝/딥러닝 실무 능력 배양 \no LangChain + RAG 를 활용한 LLM 문서검색 챗봇 구축 \no SAM2 등 멀티모달 AI 모델 활용 실습 \no MCP(Model Context Protocol) 기반 AI 모델 통합 및 API 설계 \no FastAPI 를 이용한 AI 서비스 상용화 프로젝트 완성 \no 포트폴리오 및 취업 역량 강화 \n \n2. 커리큘럼 요약 \n월 \n주차 \n주요 주제 \n실습 키워드 \n1 월 1~4 주 \nPython & PyTorch 기초 \nPython, Numpy, Pandas, 선형회귀 \n2 월 5~8 주 \nPyTorch 딥러닝 기본 \nMLP, CNN, 분류 모델 구현 \n3 월 9~12 주 LangChain + RAG 활용 \nGPT API, LangChain, FAISS, 문서 \n검색 \n4 월 13~16 주 멀티모달 AI 실습 (SAM2 \n포함)'

In [24]:
# 5단계, 검색기 생성

# 1. k=1로 설정: 아주 핵심적인 정보 하나만 딱 본다.
# retriever_1 = vectorstore.as_retriever(search_kwargs={"k": 1})
# 2. k=5로 설정: 주변 맥락까지 폭넓게 읽어서 풍부하게 답변한다.
# retriever_5 = vectorstore.as_retriever(search_kwargs={"k": 5})
# 3. 기본값: k=4
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
retriever.invoke("프로젝트")

[Document(id='f6711e79-9eaa-4ffc-95cf-e02981c56f9a', metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-08-07T08:56:01+09:00', 'source': './documents/ai커리큘럼.pdf', 'file_path': './documents/ai커리큘럼.pdf', 'total_pages': 3, 'format': 'PDF 1.5', 'title': '', 'author': 'Admin', 'subject': '', 'keywords': '', 'moddate': '2025-08-07T08:56:01+09:00', 'trapped': '', 'modDate': "D:20250807085601+09'00'", 'creationDate': "D:20250807085601+09'00'", 'page': 2}, page_content='\uf0b7 \n완성된 AI 프로젝트 2~3 건 (GitHub 공개) \n\uf0b7 \nREST API 서버 및 문서 \n\uf0b7 \n포트폴리오 발표 자료 및 데모 영상 \n\uf0b7 \n기술 이력서 및 자기소개서 템플릿 \n\uf0b7 \n모의 면접 및 실무 발표 시뮬레이션 \n \n7. 부록: 취업 전략 및 팁 \n\uf0b7 \nAI 직무 맞춤형 이력서 작성법 \n\uf0b7 \nGitHub 프로젝트 관리법 \n\uf0b7 \n기술 면접 예상 질문 및 답변 가이드')]

In [25]:
#  6단계, 프롬프트 생성
prompt_template = PromptTemplate.from_template(
    """
    귀하는 제공된 참고 문헌을 바탕으로 질문에 답하는 정보 분석 전문가입니다. 
    답변 시 반드시 제시된 문맥(Context) 내의 정보만을 활용하십시오. 
    만약 주어진 자료만으로 답변이 어렵다면, 추측하지 말고 
    '제공된 정보로는 확인이 불가능하다'고 명확히 밝히십시오. 
    모든 응답은 한국어로 작성합니다.

    #Context: 
    {context}
    
    #Question:
    {question}
    
    #Answer:
    """
)

In [26]:
# 7단계, LLM 생성
llm = ChatOpenAI(
    model_name="gpt-5.4-nano",
    temperature=0
)

In [28]:
# 8단계, 체인 생성

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

In [29]:
question = "LangChain 체인 설계는 몇 주차에 배우는가?"
answer = chain.invoke(question)
print(answer)

제공된 문맥에 따르면 **LangChain 체인 설계 및 기본 체인 구현은 10 주차에 배웁니다.**


In [5]:
# docker run -d --name redis-stack -p 6380:6379 -p 8001:8001 redis/redis-stack-server:latest
# docker exec -it redis-stack redis-cli
from langchain.globals import set_llm_cache
from langchain_community.cache import RedisSemanticCache
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader

# 1. 캐시용 임베딩 모델
cache_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sbert-nli")

# 2. 시맨틱 캐시 설정 (Docker로 띄운 Redis Stack 연결)
set_llm_cache(RedisSemanticCache(
    redis_url="redis://localhost:6380",
    embedding=cache_embeddings,
    score_threshold=0.1  # RAG는 답변의 정확도가 중요하므로 임계값을 보수적으로(낮게) 설정
))

# 단계 1: 문서 로드(Load Documents)
FILE_PATH = "./documents/ai커리큘럼.pdf"
loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()
print(f"문서의 페이지수: {len(docs)}")

# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)
print(f"분할된 청크의수: {len(split_documents)}")

# 단계 3: 임베딩(Embedding) 생성
# 우리 컴퓨터 내에서 모델을 돌리므로 데이터가 외부로 나가지 않음
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sbert-nli", # 한국어 성능이 검증된 오픈소스 모델
    model_kwargs={'device': 'cpu'}    # GPU가 있다면 'cuda'
)

# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어 생성
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

# 단계 5: 검색기(Retriever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성한다.
# 1. k=1로 설정: 아주 핵심적인 정보 하나만 딱 본다.
# retriever_1 = vectorstore.as_retriever(search_kwargs={"k": 1})
# 2. k=5로 설정: 주변 맥락까지 폭넓게 읽어서 풍부하게 답변한다.
# retriever_5 = vectorstore.as_retriever(search_kwargs={"k": 5})
# 3. 기본값: k=4
retriever = vectorstore.as_retriever()

# 단계 6: 프롬프트 생성(Create Prompt)
prompt = PromptTemplate.from_template(
    """귀하는 제공된 참고 문헌을 바탕으로 질문에 답하는 정보 분석 전문가입니다. 
    답변 시 반드시 제시된 문맥(Context) 내의 정보만을 활용하십시오. 
    만약 주어진 자료만으로 답변이 어렵다면, 추측하지 말고 
    '제공된 정보로는 확인이 불가능하다'고 명확히 밝히십시오. 
    모든 응답은 한국어로 작성합니다.

#Context: 
{context}

#Question:
{question}

#Answer:"""
)

# 단계 7: 언어모델(LLM) 생성
llm = ChatOpenAI(model_name="gpt-5.4-mini", temperature=0)

# 단계 8: 체인(Chain) 생성
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "LangChain 체인 설계는 몇 주차에 배우는가?"
response = chain.invoke(question)
print(response)

문서의 페이지수: 3
분할된 청크의수: 5
LangChain 체인 설계는 **10주차**에 배웁니다.


In [6]:
question = "LangChain 체인 설계는 몇 주차에 배우는가?"
response = chain.invoke(question)
print(response)

LangChain 체인 설계는 **10주차**에 배웁니다.


In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# docker run -d --name redis-stack -p 6379:6379 -p 8001:8001 redis/redis-stack-server:latest
# docker exec -it redis-stack redis-cli
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.globals import set_llm_cache
from langchain_community.cache import RedisSemanticCache


from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# OpenAI 대신 로컬 모델 사용
from langchain_huggingface import HuggingFaceEmbeddings

# 1. 캐시용 임베딩 모델 (RAG용 OpenAIEmbeddings와 별개로 로컬 모델 사용 가능)
cache_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sbert-nli")

# 2. 시맨틱 캐시 설정 (Docker로 띄운 Redis Stack 연결)
set_llm_cache(RedisSemanticCache(
    redis_url="redis://localhost:6380",
    embedding=cache_embeddings,
    score_threshold=0.1  # RAG는 답변의 정확도가 중요하므로 임계값을 보수적으로(낮게) 설정
))

# 단계 1: 문서 로드(Load Documents)
FILE_PATH = "./documents/공간 통계 기법을 적용한 기상 데이터 기반 태양광 발전량 예측 모델 연구.pdf"
loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()
print(f"문서의 페이지수: {len(docs)}")

# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)
print(f"분할된 청크의수: {len(split_documents)}")

# 단계 3: 임베딩(Embedding) 생성
# 우리 컴퓨터 내에서 모델을 돌리므로 데이터가 외부로 나가지 않음
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sbert-nli", # 한국어 성능이 검증된 오픈소스 모델
    model_kwargs={'device': 'cpu'}    # GPU가 있다면 'cuda'
)

# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어 생성
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

# 단계 5: 검색기(Retriever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성한다.
# 1. k=1로 설정: 아주 핵심적인 정보 하나만 딱 본다.
# retriever_1 = vectorstore.as_retriever(search_kwargs={"k": 1})
# 2. k=5로 설정: 주변 맥락까지 폭넓게 읽어서 풍부하게 답변한다.
# retriever_5 = vectorstore.as_retriever(search_kwargs={"k": 5})
# 3. 기본값: k=4
retriever = vectorstore.as_retriever()

# 단계 6: 프롬프트 생성(Create Prompt)
prompt = PromptTemplate.from_template(
    """귀하는 제공된 참고 문헌을 바탕으로 질문에 답하는 정보 분석 전문가입니다. 
    답변 시 반드시 제시된 문맥(Context) 내의 정보만을 활용하십시오. 
    만약 주어진 자료만으로 답변이 어렵다면, 추측하지 말고 
    '제공된 정보로는 확인이 불가능하다'고 명확히 밝히십시오. 
    모든 응답은 한국어로 작성합니다.

#Context: 
{context}

#Question:
{question}

#Answer:"""
)

# 단계 7: 언어모델(LLM) 생성
llm = ChatOpenAI(model_name="gpt-4.1-mini", temperature=0)

# 단계 8: 체인(Chain) 생성
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
# question = """
# 병렬 처리 기반 크리깅을 통해 산출한 발전소 위치의 추정 기상 데이터를 입력으로 적용한 경우 
# DNN의 2시간 후 예측에서 각 평가지표 별로 얼마나 개선되었나?
# """
question = """
MSE variation with respect to the number of sampling points 그래프에서 
어떤 모델끼리 비슷한가(겹쳐있는가)?
"""

response = chain.invoke(question)
print(response)

문서의 페이지수: 12
분할된 청크의수: 70
제공된 정보에 따르면, MSE variation with respect to the number of sampling points 그래프에 대한 구체적인 모델 간 비교나 어떤 모델들이 비슷하거나 겹쳐 있다는 직접적인 언급은 없습니다. 다만, 여러 베리오그램 모델(Linear, Spherical, Exponential, Power)과 IDW 모델의 MSE 값이 비교되었고, Spherical 모델이 가장 낮은 MSE를 보여 우수한 성능을 보였다는 내용은 있습니다.

하지만 "MSE variation with respect to the number of sampling points" 그래프에서 어떤 모델들이 비슷하거나 겹쳐 있는지에 대한 구체적인 설명이나 데이터는 제공된 문맥 내에 포함되어 있지 않습니다.

따라서,  
**제공된 정보로는 MSE variation with respect to the number of sampling points 그래프에서 어떤 모델끼리 비슷하거나 겹쳐 있는지 확인이 불가능하다**고 답변드립니다.


### RAG-Anything
- 다양한 형식의 문서를 분석하여 그래프 기반의 멀티모달 지식 베이스를 구축하고, 이를 통해 정교한 질의응답을 수행하는 차세대 RAG 프레임워크이다.

1. Multi-modal Content Parsing
> 1) Hierarchical Text Extraction: 문서의 제목, 본문 등 계층적 구조를 유지하며 텍스트 추출
> 2) Image Caption & Metadata: 이미지 내 객체를 감지하고 캡션을 생성하며 메타데이터 추출
> 3) LaTeX Equation Recognition: 복잡한 수식을 인식하여 기계가 읽을 수 있는 LaTeX 형식으로 변환
> 4) Table Structure Parsing: 표의 행과 열 관계를 유지하며 데이터 구조화

2. Graph-based Multi-modal Knowledge Grounding
> 1) 텍스트 정보 처리: LLM을 통해 텍스트 내 주요 개념과 이들 간의 관계 파악
> 2) 지식 그래프 생성: 추출된 정보를 노드와 엣지로 구성된 그래프 형태로 저장
> 3) VLM/LLM 프로세서: 비전 모델(VLM)을 사용하여 텍스트와 이미지/표 사이의 논리적 연결 고리 생성
> 4) Merged KG: 여러 문서에서 추출된 개별 그래프들을 하나의 거대한 지식 그래프로 통합

3. Hybrid Query & Retrieval
> 1) Graph-based Retrieval: 지식 그래프를 탐색하여 질문과 관련된 개념적 관계망 추적
> 2) Embedding-based Retrieval: 벡터 DB를 통해 질문과 유사한 텍스트 청크 및 이미지 정보를 검색
> 3) 정보 통합: 그래프 검색 결과와 벡터 검색 결과를 결합하여 LLM에게 전달

- 표의 방향이 역방향인 경우 분석 능력이 감소할 수 있으며, 이미지보다 텍스트를 우선시하는 경향이 있다.  
<hr>
<div style="text-align:center">
    <h1>
        <a href="https://github.com/HKUDS/RAG-Anything/tree/main">Rag-Anything Git-hub</a>
    </h1>
</div>
<img src="./images/rag_anything_framework.png">

`▼ 실행중인 노트북파일 모두 종료 후 CMD 관리자 권한으로 설치`

In [ ]:
# %pip install raganything

In [ ]:
# %pip install uv

In [ ]:
# cuda 환경
# %uv pip install -U "mineru[all]"

In [ ]:
# cuda or cpu 환경
# %pip install docling --ignore-installed pydantic-core

In [4]:
import asyncio
import os
from functools import partial
from dotenv import load_dotenv
import numpy as np

from raganything import RAGAnything, RAGAnythingConfig
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.utils import EmbeddingFunc
from langchain_huggingface import HuggingFaceEmbeddings

# CPU 사용 강제 설정
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# .env 파일 로드
load_dotenv()

async def main():
    # 환경 변수에서 API 키 가져오기
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("에러: .env 파일에 OPENAI_API_KEY가 설정되어 있지 않습니다.")
        return

    # 1. RAGAnything 설정
    config = RAGAnythingConfig(
        # rag_storage
        # RAG 시스템의 영구 기억장치
        # 문서 분석의 결과를 저장하기 때문에 동일한 질문을 할 때 문서를 다시 읽을 필요 없다.
        working_dir="./rag_storage",
        # parser="mineru", 
        parser="docling",
        parse_method="auto",
        enable_image_processing=True,
        enable_table_processing=True,
        enable_equation_processing=True,
    )

    # 2. LLM 모델 함수 (gpt-5.4-nano)
    def llm_model_func(prompt, system_prompt=None, history_messages=[], **kwargs):
        return openai_complete_if_cache(
            "gpt-5.4-nano",
            prompt,
            system_prompt=system_prompt,
            history_messages=history_messages,
            api_key=api_key,
            **kwargs,
        )

    # 3. 비전 모델 함수 (gpt-5.4-mini)
    def vision_model_func(prompt, system_prompt=None, history_messages=[], image_data=None, messages=None, **kwargs):
        if messages:
            return openai_complete_if_cache("gpt-5.4-mini", "", messages=messages, api_key=api_key, **kwargs)
        elif image_data:
            return openai_complete_if_cache(
                "gpt-5.4-mini", "",
                messages=[
                    {"role": "system", "content": system_prompt} if system_prompt else None,
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}},
                        ],
                    }
                ],
                api_key=api_key,
                **kwargs,
            )
        else:
            return llm_model_func(prompt, system_prompt, history_messages, **kwargs)

    # 4. 로컬 HuggingFace 임베딩 설정 (ko-sbert-nli)
    hf_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sbert-nli")
    
    async def local_embedding_func_wrapper(texts):
        # 리스트 형태로 임베딩 추출
        embeddings = hf_embeddings.embed_documents(texts)
        # 내부적으로 size함수를 사용하기 때문에 ndarray로 변환
        return np.array(embeddings)

    embedding_func = EmbeddingFunc(
        embedding_dim=768, # ko-sbert 모델의 출력 차원
        max_token_size=512,
        func=local_embedding_func_wrapper
    )

    # 5. RAGAnything 초기화
    rag = RAGAnything(
        config=config,
        llm_model_func=llm_model_func,
        vision_model_func=vision_model_func,
        embedding_func=embedding_func,
    )

    # 6. 문서 처리
    FILE_PATH = "./documents/공간 통계 기법을 적용한 기상 데이터 기반 태양광 발전량 예측 모델 연구.pdf"
    
    if not os.path.exists(FILE_PATH):
        print(f"파일을 찾을 수 없습니다: {FILE_PATH}")
        return

    print("문서 분석 및 멀티모달 지식 그래프 구축 시작...")
    await rag.process_document_complete(
        file_path=FILE_PATH,
        # PDF 내에서 추출된 표(Table), 그림(Figure), 차트 이미지 파일들이 저장된다.
        output_dir="./rag_output",
        parse_method="auto"
    )

    # 7. 질의 실행
    question = """
    MSE variation with respect to the number of sampling points 그래프에서 
    어떤 모델끼리 비슷한가(겹쳐있는가)?
    """

    print(f"\n[질문]: {question}")
    
    # 텍스트 쿼리 실행 (aquery 사용)
    result = await rag.aquery(
        question,
        mode="hybrid"
    )
    print(f"\n[분석 결과]:\n{result}")

try:
    # 파이썬 환경
    # asyncio.run(main())
    
    # 주피터 노트북 환경
    await main()
    
except Exception as e:
    print(f"실행 중 오류 발생: {e}")

INFO: Created working directory: ./rag_storage
INFO: RAGAnything initialized with config:
INFO:   Working directory: ./rag_storage
INFO:   Parser: docling
INFO:   Parse method: auto
INFO:   Multimodal processing - Image: True, Table: True, Equation: True
INFO:   Max concurrent files: 1


문서 분석 및 멀티모달 지식 그래프 구축 시작...


INFO: Parser 'docling' installation verified
INFO: Initializing LightRAG with parameters: {'working_dir': './rag_storage'}
INFO: [] Created new empty graph file: ./rag_storage\graph_chunk_entity_relation.graphml
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './rag_storage\\vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './rag_storage\\vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './rag_storage\\vdb_chunks.json'} 0 data
INFO: [] Process 11796 KV load full_docs with 0 records
INFO: [] Process 11796 KV load text_chunks with 0 records
INFO: [] Process 11796 KV load full_entities with 0 records
INFO: [] Process 11796 KV load full_relations with 0 records
INFO: [] Process 11796 KV load entity_chunks with 0 records
INFO: [] Process 11796 KV load relation_chunks with 0 records
INFO: [] Process 11796 KV load llm_response_cache 


[질문]: 
    MSE variation with respect to the number of sampling points 그래프에서 
    어떤 모델끼리 비슷한가(겹쳐있는가)?
    


INFO:  == LLM cache == saving: hybrid:keywords:80185a0cd5ed011be55b31df979dfeed
INFO: Query nodes: MSE, sampling points, 그래프에서, 모델끼리 비슷한가, 겹쳐있는가 (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 332 relations
INFO: Query edges: MSE variation, Number of sampling points, Model comparison, Graph interpretation (top_k:40, cosine:0.2)
INFO: Global query: 40 entites, 40 relations
INFO: Raw search results: 66 entities, 341 relations, 0 vector chunks
INFO: After truncation: 66 entities, 119 relations
INFO: Selecting 22 from 22 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 119 relations
INFO: Round-robin merged chunks: 22 -> 22 (deduplicated 0)
INFO: Final context: 66 entities, 119 relations, 12 chunks
INFO: Final chunks S+F/O: E3/1 E5/2 E3/3 E1/4 E1/5 E3/6 E17/7 E1/8 E1/9 E5/10 E11/11 E4/12
INFO: Found 5 image path matches in prompt
INFO: Processed 5 images for VLM
INFO: VLM enhanced query completed



[분석 결과]:
## 그래프에서 서로 비슷하게 겹쳐 보이는 모델

**MSE variation with respect to the number of sampling points** 그래프에서는 다음 모델들이 서로 비슷하게 움직이며 많이 겹쳐 보입니다.

- **Linear Model**
- **Spherical Model**
- **Exponential Model**

이 세 모델은 그래프의 대부분 구간에서 **서로 가까운 MSE 값을 유지**하며 거의 겹치듯 따라갑니다. 특히 **Spherical Model**은 이들 중에서 대체로 가장 낮거나 비슷한 수준을 보입니다.

반면에,

- **Power Model**은 다른 모델들보다 **더 높게 위치**하고,
- **IDW**는 전반적으로 **더 높은 MSE 영역**에 있어 다른 모델들과는 덜 겹칩니다.

즉, 가장 비슷하게 보이는 조합은 **Linear / Spherical / Exponential**이고, 그중에서도 **Linear와 Exponential**이 특히 가까운 구간이 많습니다.

### References

- [1] 공간 통계 기법을 적용한 기상 데이터 기반 태양광 발전량 예측 모델 연구.pdf
